In [1]:
import pandas as pd
import numpy as np
import nfl_data_py as nfl

from ELO import ELO

# Team Level Data

In [2]:
### LOAD PLAY-BY-PLAY DATA
START_YEAR = 2008
END_YEAR = 2024
WEEK = 15

pbp = nfl.import_pbp_data(list(range(START_YEAR, END_YEAR + 1)))
pbp["posteam"] = pbp["posteam"].replace({"STL": "LA", "SD": "LAC", "OAK": "LV"})
pbp["defteam"] = pbp["defteam"].replace({"STL": "LA", "SD": "LAC", "OAK": "LV"})

2008 done.
2009 done.
2010 done.
2011 done.
2012 done.
2013 done.
2014 done.
2015 done.
2016 done.
2017 done.
2018 done.
2019 done.
2020 done.
2021 done.
2022 done.
2023 done.
2024 done.
Downcasting floats.


In [3]:
### OFFENSIVE METRICS
team_offensive_data = pbp.groupby(["posteam", "season", "week"]).agg({
    # General Stats
    "drive": "nunique",                  # Number of drives
    "ydstogo": "mean",                   # Average yards to go
    "yards_gained": ["sum", "mean"],     # Total / average yards gained

    # Passing Stats
    "passing_yards": ["sum", "mean"],    # Total / average passing yards
    "air_yards": ["sum", "mean"],        # Total / average air yards
    "first_down_pass": "sum",            # Total first down passes
    "complete_pass": "sum",              # Total complete passes
    "incomplete_pass": "sum",            # Total incomplete passes
    "cpoe": "mean",                      # Average completion percentage over expectation
    "qb_dropback": "sum",                # Total dropbacks
    "qb_scramble": "sum",                # Total scrambles

    # Rushing Stats
    "rushing_yards": ["sum", "mean"],    # Total / average rushing yards
    "first_down_rush": "sum",            # Total first down rushes
    "rush_attempt": "sum",               # Total rush attempts

    # Receiving Stats
    "receiving_yards": ["sum", "mean"],  # Total / average receiving yards
    "yards_after_catch": ["sum", "mean"],# Total / average YAC

    # Scoring Stats
    "posteam_score": "max",              # Final score
    "touchdown": "sum",                  # Total touchdowns
    "pass_touchdown": "sum",             # Total passing touchdowns
    "rush_touchdown": "sum",             # Total rushing touchdowns

    # Efficiency Stats
    "epa": ["sum", "mean"],              # Total / average EPA
    "first_down_penalty": "sum",         # Total first down penalties
    "third_down_converted": "sum",       # Total third downs converted
    "third_down_failed": "sum",          # Total third downs failed
    "fourth_down_converted": "sum",      # Total fourth downs converted
    "fourth_down_failed": "sum",         # Total fourth downs failed

    # Turnovers/Negative Plays
    "penalty": "sum",                    # Total penalties
    "penalty_yards": "sum",              # Total penalty yards
    "interception": "sum",               # Total interceptions
    "fumble": "sum",                     # Total fumbles
    "fumble_lost": "sum",                # Total fumbles lost
    "tackled_for_loss": "sum",           # Total TFLs
    "qb_hit": "sum",                     # Total QB hits
    "sack": "sum"                        # Total sacks
})
team_offensive_data.columns = ['_'.join(col).strip() for col in team_offensive_data.columns]
team_offensive_data = pd.DataFrame(team_offensive_data).reset_index()
drive_offensive_data = pbp.groupby(["posteam", "season", "week", "drive"])[["drive_play_count",
                                                                            "drive_time_of_possession",
                                                                            "drive_inside20",
                                                                            "drive_ended_with_score"]].first().reset_index()
drive_offensive_data["drive_time_of_possession"] = (
    drive_offensive_data["drive_time_of_possession"]
    .str.split(":")
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
drive_offensive_data = drive_offensive_data.groupby(["posteam", "season", "week"]).agg({
    "drive_play_count": "sum",
    "drive_time_of_possession": "sum",
    "drive_inside20": "sum",
    "drive_ended_with_score": "sum"
}).rename(columns={"drive_play_count": "total_plays",
                   "drive_time_of_possession": "total_time_of_posession"}).reset_index()
team_offensive_data = pd.merge(team_offensive_data,
                            drive_offensive_data,
                            on = ["posteam", "season", "week"],
                            how = "left")
team_offensive_data["qb_dropback_percentage"] = team_offensive_data["qb_dropback_sum"] / team_offensive_data["total_plays"]
team_offensive_data["qb_scramble_percentage"] = team_offensive_data["qb_scramble_sum"] / team_offensive_data["total_plays"]
team_offensive_data["first_down_sum"] = team_offensive_data["first_down_pass_sum"] + team_offensive_data["first_down_rush_sum"] + team_offensive_data["first_down_penalty_sum"]
team_offensive_data["third_down_conversion_rate"] = team_offensive_data["third_down_converted_sum"] / (team_offensive_data["third_down_converted_sum"] + team_offensive_data["third_down_failed_sum"])
team_offensive_data["net_turnovers"] = team_offensive_data["interception_sum"] + team_offensive_data["fumble_lost_sum"]
team_offensive_data["drive_scoring_percentage"] = team_offensive_data["drive_ended_with_score"] / team_offensive_data["drive_nunique"] 
team_offensive_data = team_offensive_data.rename(
    columns = lambda col: col + "_off" if col not in ["posteam", "season", "week"] else col
)

In [4]:
### DEFENSIVE METRICS
team_mapping = pbp[["season", "week", "posteam", "defteam"]].drop_duplicates().dropna().reset_index(drop = True)
team_offensive_data = pd.merge(team_offensive_data, team_mapping, on = ["season", "week", "posteam"], how = "left")
team_defensive_data = team_offensive_data.copy()
team_defensive_data = team_defensive_data.rename(
    columns = lambda col: col.replace("_off", "_def") if "_off" in col else col
)

In [5]:
### MERGE OFFENSIVE AND DEFENSIVE METRICS
team_data = pd.merge(
    team_offensive_data,
    team_defensive_data,
    left_on = ["season", "week", "defteam"],
    right_on = ["season", "week", "posteam"],
    suffixes = ("", "_def")
)
team_data = team_data.drop(columns = ["posteam_def", "defteam_def", "defteam"])

In [7]:
### ROLLING AVERAGES
team_data_rolling = pd.DataFrame()

# Process data for each team
for team in team_data["posteam"].unique():
    individual_team_data = team_data[team_data["posteam"] == team].sort_values(by=["season", "week"])
    
    # Calculate rolling averages for each feature and overwrite them
    for feature in [col for col in team_data.columns if col not in ["posteam", "season", "week"]]:
        individual_team_data[feature] = (
            individual_team_data[feature]
            .rolling(window = 10, min_periods = 1, closed = "left")
            .mean()
        )
    
    # Append the processed team data to the final DataFrame
    team_data_rolling = pd.concat([team_data_rolling, individual_team_data])

# ELO Data

In [8]:
### ELO MODEL
elo_model = ELO(2010, END_YEAR, WEEK)
elo_model.read_games()
elo_model.forecast()
elo_df = elo_model.games_df

In [9]:
### MERGE DATASETS
home_team_features = team_data_rolling.rename(columns = lambda col: f"{col}_home" if col not in ["posteam", "season", "week"] else col)
data = pd.merge(elo_df, home_team_features, how = "left", left_on = ["home_team", "season", "week"], right_on = ["posteam", "season", "week"])
data = data.drop(columns = ["posteam"])

away_team_features = team_data_rolling.rename(columns = lambda col: f"{col}_away" if col not in ["posteam", "season", "week"] else col)
data = pd.merge(data, away_team_features, how = "left", left_on = ["away_team", "season", "week"], right_on = ["posteam", "season", "week"])
data = data.drop(columns = ["posteam"])

# Custom Features

In [10]:
def current_win_pctg(df):
    """
    Computes current win percentage features for home and away teams.

    Inputs:
        - df: pd.DataFrame, Input data
    Outputs:
        - df: pd.DataFrame, Output data with new features
    """
    temp = {team: {"wins": 0,
                   "games": 0,
                   "season": df["season"].min()} for team in df["home_team"].unique()}

    current_win_pctg_home = []
    current_win_pctg_away = []

    for row in df.itertuples():
        for team in [row.home_team, row.away_team]:
            if temp[team]["season"] != row.season:
                temp[team]["wins"] = 0
                temp[team]["games"] = 0
            temp[team]["season"] = row.season
        current_win_pctg_home.append(
            temp[row.home_team]["wins"] / temp[row.home_team]["games"]
            if temp[row.home_team]["games"] > 0 else 0.0
        )
        current_win_pctg_away.append(
            temp[row.away_team]["wins"] / temp[row.away_team]["games"]
            if temp[row.away_team]["games"] > 0 else 0.0
        )
        if row.result > 0:
            winner, loser = row.home_team, row.away_team
        elif row.result < 0:
            winner, loser = row.away_team, row.home_team
        else:
            continue
        temp[winner]["wins"] += 1
        temp[winner]["games"] += 1
        temp[loser]["games"] += 1

    df["current_win_pctg_home"] = current_win_pctg_home
    df["current_win_pctg_away"] = current_win_pctg_away

    return df

In [11]:
def last_win_pctg(df):
    """
    Computes last season win percentage features for home and away teams.

    Inputs:
        - df: pd.DataFrame, Input data
    Outputs:
        - df: pd.DataFrame, Output data with new features
    """
    temp = {team: {"wins": 0,
                   "games": 0,
                   "last_win_pctg": 0,
                   "season": df["season"].min()} for team in df["home_team"].unique()}

    last_win_pctg_home = []
    last_win_pctg_away = []

    for row in df.itertuples():
        for team in [row.home_team, row.away_team]:
            if row.season == data["season"].min():
                temp[team]["last_win_pctg"] = 0.5
            if temp[team]["season"] != row.season:
                temp[team]["last_win_pctg"] = (
                    temp[team]["wins"] / temp[team]["games"]
                )
                temp[team]["wins"] = 0
                temp[team]["games"] = 0
            temp[team]["season"] = row.season
        last_win_pctg_home.append(temp[row.home_team]["last_win_pctg"])
        last_win_pctg_away.append(temp[row.away_team]["last_win_pctg"])
        if row.result > 0:
            winner, loser = row.home_team, row.away_team
        elif row.result < 0:
            winner, loser = row.away_team, row.home_team
        else:
            continue
        temp[winner]["wins"] += 1
        temp[winner]["games"] += 1
        temp[loser]["games"] += 1

    df["last_win_pctg_home"] = last_win_pctg_home
    df["last_win_pctg_away"] = last_win_pctg_away
    
    return df

In [12]:
def streak(df):
    """
    Computes current win streak features for home and away teams.

    Inputs:
        - df: pd.DataFrame, Input data
    Outputs:
        - df: pd.DataFrame, Output data with new features
    """
    temp = {team: 0 for team in df["home_team"].unique()}

    streak_home = []
    streak_away = []

    for row in df.itertuples():
        streak_home.append(temp[row.home_team])
        streak_away.append(temp[row.away_team])
        if row.result > 0:
            winner, loser = row.home_team, row.away_team
        elif row.result < 0:
            winner, loser = row.away_team, row.home_team
        else:
            continue
        temp[winner] += 1
        temp[loser] = 0

    df["streak_home"] = streak_home
    df["streak_away"] = streak_away

    return df

In [13]:
def ats_pctg(df, n = 10):
    """
    Computes winning percentage against the spread for home and away teams.

    Inputs:
        - df: pd.DataFrame, Input data
        - n: int, Number of previous games to calculate
    Outputs:
        - df: pd.DataFrame, Output data with new features
    """
    temp = {team: [0] * n for team in df["home_team"].unique()}

    ats_pctg_home = []
    ats_pctg_away = []

    for row in df.itertuples():
        ats_pctg_home.append(sum(temp[row.home_team]) / n)
        ats_pctg_away.append(sum(temp[row.away_team]) / n)
        if row.result > row.spread_line:
            temp[row.home_team].pop()
            temp[row.home_team].insert(0, 1)
            temp[row.away_team].pop()
            temp[row.away_team].insert(0, 0)
        elif row.result < row.spread_line:
            temp[row.home_team].pop()
            temp[row.home_team].insert(0, 0)
            temp[row.away_team].pop()
            temp[row.away_team].insert(0, 1)
        else:
            continue

    df["ats_pctg_home"] = ats_pctg_home
    df["ats_pctg_away"] = ats_pctg_away

    return df

In [14]:
def spread_differential(df, n = 10):
    """
    Computes spread differential for home and away teams.

    Inputs:
        - df: pd.DataFrame, Input data
        - n: int, Number of previous games to calculate
    Outputs:
        - df: pd.DataFrame, Output data with new features
    """
    temp = {team: [0] * n for team in df["home_team"].unique()}

    spread_differential_home = []
    spread_differential_away = []

    for row in df.itertuples():
        spread_differential_home.append(sum(temp[row.home_team]))
        spread_differential_away.append(sum(temp[row.away_team]))
        temp[row.home_team].pop()
        temp[row.home_team].insert(0, row.result - row.spread_line)
        temp[row.away_team].pop()
        temp[row.away_team].insert(0, row.spread_line - row.result)

    df["spread_differential_home"] = spread_differential_home
    df["spread_differential_away"] = spread_differential_away

    return df

In [15]:
def qbr(df):
    """
    Calculates and updates rolling QBR for each game based on prior performance and game outcomes.

    Inputs: 
        - df: pd.DataFrame, Input data
    Outputs:
        - df: pd.DataFrame, Output data with new features

    Assigns a baseline QBR of 35.0 for rookies.
    Updates rolling QBR with weighted averages based on past performance:
        o Adjust QBR toward league average (50.0) for QBs with 10-100 career starts.
        o Updates QBR post-game with 90/10 weighting of old QBR and current performance.
    """
    qb_dictionary = {}
    qbr_home = []
    qbr_away = []

    url = "https://github.com/nflverse/nflverse-data/releases/download/espn_data/qbr_week_level.parquet"
    qbr_df = pd.read_parquet(url).loc[lambda x: x.season.isin([2006, df.season.max()])].reset_index(drop = True)

    # Update QB dictionary with average historical performance since 2006
    for row in qbr_df.itertuples():
        if row.season >= df["season"].min():
            break
        else:
            qb_name = row.name_display
            qb_performance = row.qbr_total
            qb_season = row.season
            if qb_name not in qb_dictionary:
                qb_dictionary[qb_name] = {"qbr": qb_performance,
                                          "starts": 1,
                                          "season": qb_season}
            else:
                qb_dictionary[qb_name]["qbr"] = (
                    (qb_dictionary[qb_name]["qbr"] * qb_dictionary[qb_name]["starts"] + qb_performance) / (qb_dictionary[qb_name]["starts"] + 1)
                )
                qb_dictionary[qb_name]["starts"] += 1
                qb_dictionary[qb_name]["season"] = qb_season

    # Iterate through games and update QBR for each QB
    for row in df.itertuples():
        game_id = str(row.espn)
        home_qb_name = row.home_qb_name
        away_qb_name = row.away_qb_name

        # Handle home QB performance at the start of game
        if home_qb_name not in qb_dictionary:
            home_qb_performance = 50.0
        else:
            if row.season != qb_dictionary[home_qb_name]["season"]:
                if 10 < qb_dictionary[home_qb_name]["starts"] < 100:
                    qb_dictionary[home_qb_name]["qbr"] = (
                        0.75 * qb_dictionary[home_qb_name]["qbr"] + 0.25 * 50.0
                    )
            home_qb_performance = qb_dictionary[home_qb_name]["qbr"]

        # Handle away QB performance at the start of game
        if away_qb_name not in qb_dictionary:
            away_qb_performance = 50.0
        else:
            if row.season != qb_dictionary[away_qb_name]["season"]:
                if 10 < qb_dictionary[away_qb_name]["starts"] < 100:
                    qb_dictionary[away_qb_name]["qbr"] = (
                        0.75 * qb_dictionary[away_qb_name]["qbr"] + 0.25 * 50.0
                    )
            away_qb_performance = qb_dictionary[away_qb_name]["qbr"]

        # Append home and away QB performances
        qbr_home.append(home_qb_performance)
        qbr_away.append(away_qb_performance)

        # Update QB dictionary with performance after game occurs
        matched_df = qbr_df[qbr_df["game_id"] == game_id]
        for subrow in matched_df.itertuples():
            qb_name = subrow.name_display
            qb_performance = subrow.qbr_total
            if qb_name not in qb_dictionary:
                qb_dictionary[qb_name] = {"qbr": qb_performance,
                                          "starts": 1,
                                          "season": subrow.season}
            else:
                qb_dictionary[qb_name]["qbr"] = (
                    0.9 * qb_dictionary[qb_name]["qbr"] + 0.1 * qb_performance
                )
                qb_dictionary[qb_name]["starts"] += 1
                qb_dictionary[qb_name]["season"] = subrow.season

    df["qbr_home"] = qbr_home
    df["qbr_away"] = qbr_away

    return df

In [16]:
def travel_distance(df):
    """
    Computes travel distance for away team.

    Inputs:
        - df: pd.DataFrame, Input data
    Outputs:
        - df: pd.DataFrame, Output data with new feature
    """
    travel_distance = []
    distance_matrix = pd.read_csv("data/distance_matrix.csv")
    for row in df.itertuples():
        if row.location == "Neutral":
            travel_distance.append(0.0)
        else:
            distance = distance_matrix[distance_matrix["TEAM"] == row.away_team][row.home_team].values[0]
            travel_distance.append(distance)
    df["travel_distance"] = travel_distance
    return df

In [17]:
def clean_weeks(df):
    """
    Function to clean the "week" column in a DataFrame. For all seasons 2021 and later,
    week 18 is changed to week 17. All playoff weeks are clipped to 18.

    Inputs:
        - df: pd.DataFrame, DataFrame with "season" and "week" columns
    Outputs:
        - df: pd.DataFrame, DataFrame with cleaned "week" column
    """
    df.loc[(df["season"] >= 2021) & (df["week"] == 18), "week"] = 17
    df["week"] = df["week"].clip(upper = 18)
    return df

In [18]:
def implied_probability(df):
    """
    Function to calculate implied win probabilities based on moneyline odds.

    Inputs:
        - df: pd.DataFrame, DataFrame with "moneyline_home" and "moneyline_away" columns
    Outputs:
        - df: pd.DataFrame, DataFrame with "implied_prob_home" and "implied_prob_away" columns
    """
    def calculate_implied_probability(odds):
        if odds > 0:
            return 100 / (100 + odds)
        elif odds < 0:
            return -odds / (-odds + 100)
        else:
            return None
    df["implied_prob_home"] = df["home_moneyline"].apply(calculate_implied_probability)
    df["implied_prob_away"] = df["away_moneyline"].apply(calculate_implied_probability)
    return df

In [19]:
data["is_playoff"] = data["game_type"].map(lambda x: 0 if x == "REG" else 1)
current_win_pctg(data)
last_win_pctg(data)
streak(data)
ats_pctg(data)
spread_differential(data)
qbr(data)
travel_distance(data)
clean_weeks(data)
implied_probability(data)

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,streak_away,ats_pctg_home,ats_pctg_away,spread_differential_home,spread_differential_away,qbr_home,qbr_away,travel_distance,implied_prob_home,implied_prob_away
0,2010_01_MIN_NO,2010,REG,1,2010-09-09,Thursday,20:30,MIN,9.0,NO,...,0,0.0,0.0,0.0,0.0,61.479412,48.540000,1052.317412,0.687500,0.336700
1,2010_01_MIA_BUF,2010,REG,1,2010-09-12,Sunday,13:00,MIA,15.0,BUF,...,0,0.0,0.0,0.0,0.0,50.000000,50.000000,1164.796996,0.416667,0.607843
2,2010_01_DET_CHI,2010,REG,1,2010-09-12,Sunday,13:00,DET,14.0,CHI,...,0,0.0,0.0,0.0,0.0,42.800000,50.000000,236.633320,0.736842,0.287356
3,2010_01_IND_HOU,2010,REG,1,2010-09-12,Sunday,13:00,IND,24.0,HOU,...,0,0.0,0.0,0.0,0.0,72.100000,71.903750,870.931436,0.485437,0.539171
4,2010_01_DEN_JAX,2010,REG,1,2010-09-12,Sunday,13:00,DEN,17.0,JAX,...,0,0.0,0.0,0.0,0.0,62.144444,50.000000,1468.325160,0.649123,0.375940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4012,2024_15_TB_LAC,2024,REG,15,2024-12-15,Sunday,16:25,TB,40.0,LAC,...,3,0.8,0.6,37.5,53.5,51.202590,59.658304,2151.837826,0.618321,0.423729
4013,2024_15_PIT_PHI,2024,REG,15,2024-12-15,Sunday,16:25,PIT,13.0,PHI,...,2,0.6,0.7,58.0,46.5,59.483681,59.460873,258.700813,0.726027,0.317460
4014,2024_15_GB_SEA,2024,REG,15,2024-12-15,Sunday,20:20,GB,30.0,SEA,...,0,0.5,0.5,7.5,30.0,55.989358,53.389379,1643.696888,0.423729,0.618321
4015,2024_15_CHI_MIN,2024,REG,15,2024-12-16,Monday,20:00,CHI,12.0,MIN,...,0,0.6,0.5,8.0,-7.5,67.254334,44.014037,355.344526,0.764706,0.277778


# Save Data

In [20]:
file_path = f"data/data_engineering_{START_YEAR}_{END_YEAR}_{WEEK}.csv"
data.to_csv(file_path, index = False)